<a href="https://colab.research.google.com/github/FRakhmatov/desktop-tutorial/blob/main/Real-Time_Zero-Day_Attack_Detection_Using_a_Lightweight_Cascade_Architecture_test_in_CSE-CIC-IDS2018.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Lightweight Cascade-Based Farmework for Real-time Ze-ro-Day Attack Detection
Dataset: CSE-CIC-IDS2018

# A Lightweight Cascade-Based Farmework for Real-time Ze-ro-Day Attack Detection

This repository contains Python code demonstrating a two-stage XGBoost model for zero-day network intrusion detection. The approach is designed to identify novel attack types (zero-day attacks) that were not present in the training data.

## Table of Contents
- [Introduction](#introduction)
- [Dataset](#dataset)
- [Methodology](#methodology)
- [Results](#results)
- [Setup and Usage](#setup-and-usage)

## Introduction
Traditional intrusion detection systems often struggle with zero-day attacks due to their reliance on known attack patterns. This project proposes a two-stage machine learning model using XGBoost to enhance the detection of such novel threats. The model is trained on the CSE-CIC-IDS2018 dataset, simulating a zero-day scenario by withholding certain attack types from the training set.

## Dataset
**CSE-CIC-IDS2018**: This dataset is a comprehensive collection of network traffic that includes benign and various attack scenarios. For this experiment, the dataset is split to create a zero-day scenario where specific attack types are unknown during training.

**Data Path**: `/content/sample_data/CIC_IDS_2018.csv`

**Training Attack Types (Known)**:
- DDOS attack-LOIC-UDP
- DoS attacks-Slowloris
- Brute Force -Web
- Brute Force -XSS

**Testing Attack Types (Zero-Day)**:
- Infiltration
- SQL Injection

## Methodology
The detection system consists of two stages:

1.  **Stage 1 (Recall-first XGBoost)**:
    -   **Objective**: Maximize recall to capture as many potential attacks as possible, even at the cost of some false positives.
    -   **Configuration**: `n_estimators=300`, `max_depth=6`, `learning_rate=0.1`, `subsample=0.9`, `colsample_bytree=0.9`, `scale_pos_weight` adjusted for class imbalance.
    -   **Threshold (TH1)**: `0.15` - instances with scores above this threshold are considered suspicious and passed to Stage 2.

2.  **Stage 2 (Precision-first XGBoost)**:
    -   **Objective**: Refine the suspicious instances from Stage 1 to reduce false positives and improve precision.
    -   **Configuration**: `n_estimators=400`, `max_depth=4`, `learning_rate=0.05`, `gamma=1.0`, `reg_alpha=0.5`, `reg_lambda=1.0`.
    -   **Threshold (TH2)**: `0.85` - instances with scores above this threshold are ultimately classified as attacks.

**Preprocessing**: The data undergoes `StandardScaler` normalization before being fed into the models.

## Results

The experiment was run on the CSE-CIC-IDS2018 dataset with the defined zero-day split. The performance metrics are as follows:

- **Accuracy**: `0.9999761628283145` (99.997%)
- **False Positive Rate (FPR)**: `1.0218958211273554e-05` (0.001%)
- **Latency (Inference)**: `0.0086 ms`
- **Throughput (Inference)**: `116291 req/s`
- **Memory Usage**: `0.0 MB` (during inference)

**Confusion Matrix**:
```
[[293569      3]
 [     4     83]]
```
-   **True Negatives (TN)**: 293569
-   **False Positives (FP)**: 3
-   **False Negatives (FN)**: 4
-   **True Positives (TP)**: 83

These results indicate a highly effective detection system with very low false positive and false negative rates, demonstrating strong capabilities in identifying both benign and zero-day attack instances.

## Setup and Usage

To run this code, you will need:

-   Python 3.x
-   Required Python libraries: `pandas`, `numpy`, `psutil`, `xgboost`, `sklearn` (specifically `StandardScaler` and `confusion_matrix`).

**Installation**:
1.  Clone this repository:
    ```bash
    git clone <repository-url>
    cd <repository-name>
    ```
2.  Install the necessary Python packages:
    ```bash
    pip install pandas numpy psutil xgboost scikit-learn
    ```

**Data**: Ensure the `CIC_IDS_2018.csv` file is located at `/content/sample_data/` or update the `DATA_PATH` variable in the script to the correct location of your dataset.

**Execution**:
Run the main script:
```bash
python your_script_name.py
```
(If you are running this in a Colab environment, simply execute the provided cells sequentially.)

The script will output the loading, training, and evaluation results directly to the console.

In [ ]:
import pandas as pd
import numpy as np
import time
import psutil
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier


# =====================================================
# Utilities
# =====================================================

def memory_mb():
    return psutil.Process().memory_info().rss / (1024 ** 2)


# =====================================================
# Load CSE-CIC-IDS2018 with attack-type zero-day split
# =====================================================

def load_cicids2018_zero_day(path, train_attacks, test_attacks):
    df = pd.read_csv(path)

    # Normalize labels
    df["Label"] = df["Label"].str.strip()

    # Binary label
    df["binary_label"] = df["Label"].apply(
        lambda x: 0 if x == "Benign" else 1
    )

    # Zero-day split
    is_train = (df["Label"] == "Benign") | (df["Label"].isin(train_attacks))
    is_test  = (df["Label"] == "Benign") | (df["Label"].isin(test_attacks))

    df_train = df[is_train].copy()
    df_test  = df[is_test].copy()

    # Remove non-numeric / ID fields
    drop_cols = [
        "Flow ID", "Timestamp",
        "Src IP", "Dst IP",
        "Src Port", "Dst Port",
        "Protocol", "Label"
    ]
    df_train.drop(columns=[c for c in drop_cols if c in df_train.columns], inplace=True)
    df_test.drop(columns=[c for c in drop_cols if c in df_test.columns], inplace=True)

    X_train = df_train.drop(columns=["binary_label"])
    y_train = df_train["binary_label"].values

    X_test = df_test.drop(columns=["binary_label"])
    y_test = df_test["binary_label"].values

    return X_train, y_train, X_test, y_test


# =====================================================
# MAIN
# =====================================================

if __name__ == "__main__":

    DATA_PATH = "/content/sample_data/CIC_IDS_2018.csv"   # объединённый CSV

    TRAIN_ATTACKS = [
        "DDOS attack-LOIC-UDP",
        "DoS attacks-Slowloris",
        "Brute Force -Web",
        "Brute Force -XSS"
    ]

    TEST_ATTACKS = [
    #    "Bot",
        "Infiltration",
        "SQL Injection"
    ]

    TH1 = 0.15
    TH2 = 0.85

    print("🔄 Loading CSE-CIC-IDS2018 with zero-day split...")
    X_train_raw, y_train, X_test_raw, y_test = load_cicids2018_zero_day(
        DATA_PATH,
        TRAIN_ATTACKS,
        TEST_ATTACKS
    )

    # ===============================
    # Scaling
    # ===============================
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)

    neg, pos = np.bincount(y_train)

    # ===============================
    # Stage-1 (Recall-first)
    # ===============================
    stage1 = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=neg / pos,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    print("🚀 Training Stage-1 (Recall-first)...")
    stage1.fit(X_train, y_train)

    scores1_test = stage1.predict_proba(X_test)[:, 1]
    suspicious = scores1_test > TH1

    # ===============================
    # Stage-2 (Precision-first)
    # ===============================
    stage2 = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        gamma=1.0,
        reg_alpha=0.5,
        reg_lambda=1.0,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    scores1_train = stage1.predict_proba(X_train)[:, 1]
    train_mask = scores1_train > TH1

    if len(np.unique(y_train[train_mask])) > 1:
        print("🚀 Training Stage-2 (Precision-first)...")
        stage2.fit(X_train[train_mask], y_train[train_mask])
    else:
        print("⚠️ Stage-2 skipped (single class)")
        stage2 = None

    # ===============================
    # Final decision
    # ===============================
    y_pred = np.zeros_like(y_test)

    if stage2 is not None:
        scores2 = stage2.predict_proba(X_test[suspicious])[:, 1]
        y_pred[suspicious] = (scores2 > TH2).astype(int)
    else:
        y_pred[suspicious] = 1

    # ===============================
    # Detection metrics
    # ===============================
    cm = confusion_matrix(y_test, y_pred)
    accuracy = np.mean(y_test == y_pred)
    fpr = cm[0, 1] / (cm[0, 0] + cm[0, 1])

    # ===============================
    # Runtime metrics (Inference only)
    # ===============================
    _ = stage1.predict(X_test[:100])  # warm-up

    mem_before = memory_mb()
    t0 = time.time()
    _ = stage1.predict(X_test)
    infer_time = (time.time() - t0) * 1000
    mem_after = memory_mb()

    latency = infer_time / X_test.shape[0]
    throughput = 1000 / latency if latency > 0 else 0
    memory_used = mem_after - mem_before

    # ===============================
    # Results
    # ===============================
    print("\n📊 CSE-CIC-IDS2018 ZERO-DAY RESULTS")
    print(f"Accuracy     : {accuracy:.4f}")
    print(f"FPR          : {fpr:.4f}")
    print(f"Latency      : {latency:.4f} ms")
    print(f"Throughput   : {throughput:.0f} req/s")
    print(f"Memory (MB)  : {memory_used:.2f}")
    print("Confusion Matrix:")
    print(cm)

    print("\n✅ CSE-CIC-IDS2018 zero-day experiment completed successfully.")

